# اجرای اعتبارسنجی سؤالات از طریق مدل زبانی (LLM Validation Runner)

این نوت‌بوک فایل `validation_requests.json` (خروجی مرحله‌ی بازیابی معنایی) را می‌خواند، برای هر سؤال یک Prompt مبتنی بر شواهد واقعی نظرات می‌سازد، از طریق API سازگار با OpenAI (روی Metis AI) پاسخ می‌گیرد و نتایج را برای بررسی دستی در قالب JSON و CSV ذخیره می‌کند.

> ⚠️ **نکته‌ی امنیتی:** در این فایل یک کلید API به‌صورت متنی (hardcoded) نوشته شده است. چون این کلید اکنون در فایل ذخیره و جابه‌جا شده، پیشنهاد می‌شود آن را از حساب Metis AI باطل/تعویض کنید و به‌جای نوشتن مستقیم در کد، از متغیر محیطی (environment variable) یا فایل `.env` استفاده کنید.


## بخش ۱ — تست اتصال به API و مشاهده‌ی مدل‌های در دسترس

In [ ]:
import requests

# اندپوینت لیست مدل‌ها (برای تست اتصال و اعتبار کلید)
API_URL = "https://api.metisai.ir/openai/v1/models"
API_KEY = "****"

headers = {"Authorization": f"Bearer {API_KEY}"}

response = requests.get(API_URL, headers=headers, timeout=30)

print("Status code:", response.status_code)
print("Response:")
print(response.text)


Status code: 200
Response:
{"object":"list","data":[{"id":"text-embedding-ada-002","object":"model","created":1671217299,"owned_by":"openai-internal"},{"id":"whisper-1","object":"model","created":1677532384,"owned_by":"openai-internal"},{"id":"gpt-3.5-turbo","object":"model","created":1677610602,"owned_by":"openai"},{"id":"tts-1","object":"model","created":1681940951,"owned_by":"openai-internal"},{"id":"gpt-3.5-turbo-16k","object":"model","created":1683758102,"owned_by":"openai-internal"},{"id":"gpt-4-0613","object":"model","created":1686588896,"owned_by":"openai"},{"id":"gpt-4","object":"model","created":1687882411,"owned_by":"openai"},{"id":"davinci-002","object":"model","created":1692634301,"owned_by":"system"},{"id":"babbage-002","object":"model","created":1692634615,"owned_by":"system"},{"id":"gpt-3.5-turbo-instruct","object":"model","created":1692901427,"owned_by":"system"},{"id":"gpt-3.5-turbo-instruct-0914","object":"model","created":1694122472,"owned_by":"system"},{"id":"gpt-3

## بخش ۲ — تنظیمات و بارگذاری درخواست‌های اعتبارسنجی

In [2]:
import json
import time
import requests
import pandas as pd

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------

INPUT_FILE = "validation_requests.json"
OUTPUT_JSON = "validation_results.json"
OUTPUT_CSV = "validation_manual_review.csv"

# اندپوینت chat/completions
API_URL = "https://api.metisai.ir/openai/v1/chat/completions"
API_KEY = "tpsg-2gs5I7ZZP9uiGuN82wmv3brAnoIAV50"

MODEL_NAME = "gpt-4o-mini"

# ------------------------------------------------------------
# بارگذاری درخواست‌ها
# ------------------------------------------------------------

with open(INPUT_FILE, "r", encoding="utf-8") as f:
    validation_requests = json.load(f)

print("Validation requests:", len(validation_requests))


Validation requests: 192


## بخش ۳ — ساخت Prompt بر اساس شواهد (Evidence-grounded Prompt)

In [3]:
def build_prompt(item):
    product_title = item["product_title"]
    question = item["question"]

    evidence_text = []
    for evidence in item["evidence"]:
        evidence_text.append(f"""
COMMENT ID: {evidence["comment_id"]}

Title:
{evidence["title"]}

Body:
{evidence["body"]}

Advantages:
{evidence["advantages"]}

Disadvantages:
{evidence["disadvantages"]}

Rate:
{evidence["rate"]}

Recommendation:
{evidence["recommendation_status"]}
""")
    evidence_text = "\n\n".join(evidence_text)

    prompt = f"""
شما یک دستیار تحلیل نظرات کاربران دیجی‌کالا هستید.

محصول:
{product_title}

سؤال:
{question}

در ادامه تعدادی نظر واقعی کاربران درباره همین محصول ارائه شده است.

فقط و فقط بر اساس این نظرات پاسخ بده.

اگر شواهد کافی برای پاسخ وجود ندارد، صریحاً بگو که
شواهد کافی در نظرات ارائه‌شده وجود ندارد.

هیچ اطلاعاتی را که در شواهد وجود ندارد به عنوان واقعیت بیان نکن.

در پاسخ، در صورت امکان به Comment IDهای مرتبط اشاره کن.

نظرات:

{evidence_text}
"""
    return prompt


## بخش ۴ — تابع فراخوانی API

In [4]:
def call_api(prompt):
    headers = {"Content-Type": "application/json"}
    if API_KEY:
        headers["Authorization"] = f"Bearer {API_KEY}"

    payload = {
        "model": MODEL_NAME,
        "messages": [
            {
                "role": "system",
                "content": "You are a grounded product-review question answering assistant.",
            },
            {"role": "user", "content": prompt},
        ],
        "temperature": 0,
    }

    start_time = time.perf_counter()
    response = requests.post(API_URL, headers=headers, json=payload, timeout=300)
    latency = time.perf_counter() - start_time

    response.raise_for_status()
    data = response.json()

    # پاسخ سازگار با فرمت OpenAI
    answer = data["choices"][0]["message"]["content"]

    return answer, latency, data


## بخش ۵ — اجرای اعتبارسنجی روی همه‌ی درخواست‌ها

In [5]:
results = []

for index, item in enumerate(validation_requests, start=1):
    print(f"[{index}/{len(validation_requests)}] {item['product_title']} | {item['question']}")

    prompt = build_prompt(item)

    try:
        answer, latency, raw_response = call_api(prompt)

        result = {
            "product_id": item["product_id"],
            "product_title": item["product_title"],
            "question": item["question"],
            "answer": answer,
            "latency_seconds": latency,
            "evidence": item["evidence"],
        }
        results.append(result)

        print(f"Latency: {latency:.2f}s")

    except Exception as e:
        print("ERROR:", str(e))
        results.append({
            "product_id": item["product_id"],
            "product_title": item["product_title"],
            "question": item["question"],
            "answer": None,
            "error": str(e),
            "latency_seconds": None,
            "evidence": item["evidence"],
        })


[1/192] برس مژه و ابرو کد BR01ABO | مردم بیشتر از چه چیزی در این محصول راضی بودند؟
Latency: 2.18s
[2/192] برس مژه و ابرو کد BR01ABO | ایرادهای پرتکرار این محصول چیست؟
Latency: 1.92s
[3/192] برس مژه و ابرو کد BR01ABO | خریداران درباره‌ی کیفیت این محصول چه گفته‌اند؟
Latency: 3.00s
[4/192] برس مژه و ابرو کد BR01ABO | آیا با توجه به تجربه‌ی کاربران ارزش خرید دارد؟
Latency: 2.91s
[5/192] پیلینگ صورت مدل Eli-74 | مردم بیشتر از چه چیزی در این محصول راضی بودند؟
Latency: 2.08s
[6/192] پیلینگ صورت مدل Eli-74 | ایرادهای پرتکرار این محصول چیست؟
Latency: 1.58s
[7/192] پیلینگ صورت مدل Eli-74 | خریداران درباره‌ی کیفیت این محصول چه گفته‌اند؟
Latency: 5.46s
[8/192] پیلینگ صورت مدل Eli-74 | آیا با توجه به تجربه‌ی کاربران ارزش خرید دارد؟
Latency: 2.44s
[9/192] برس ریمل مدل LM02 مجموعه 2 عددی | مردم بیشتر از چه چیزی در این محصول راضی بودند؟
Latency: 2.45s
[10/192] برس ریمل مدل LM02 مجموعه 2 عددی | ایرادهای پرتکرار این محصول چیست؟
Latency: 1.51s
[11/192] برس ریمل مدل LM02 مجموعه 2 عددی | خریداران درباره‌ی 

## بخش ۶ — ذخیره‌سازی نتایج خام (JSON)

In [6]:
with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print(f"Saved JSON: {OUTPUT_JSON}")


Saved JSON: validation_results.json


## بخش ۷ — ساخت CSV نهایی برای بررسی دستی (Manual Review)

این جدول برای هر سؤال، پاسخ مدل را در کنار متن کامل شواهدی که در اختیارش قرار گرفته می‌گذارد تا بتوان به‌صورت دستی درستی پاسخ را بررسی کرد. *(نسخه‌ی ساده‌ی CSV که در بخش قبل تولید می‌شد، در همین‌جا با نسخه‌ی کامل‌تر جایگزین می‌شود، بنابراین آن ذخیره‌ی میانی حذف شد.)*

In [7]:
csv_rows = []

for result in results:
    evidence_ids = []
    evidence_texts = []

    for evidence in result["evidence"]:
        evidence_ids.append(str(evidence["comment_id"]))

        text = (
            f"[COMMENT ID: {evidence['comment_id']}]\n"
            f"Title: {evidence['title']}\n"
            f"Body: {evidence['body']}\n"
            f"Advantages: {evidence['advantages']}\n"
            f"Disadvantages: {evidence['disadvantages']}"
        )
        evidence_texts.append(text)

    csv_rows.append({
        "product_id": result["product_id"],
        "product_title": result["product_title"],
        "question": result["question"],
        "api_answer": result["answer"],
        "evidence_comment_ids": " | ".join(evidence_ids),
        "evidence_comments": "\n\n".join(evidence_texts),
        "latency_seconds": result["latency_seconds"],
    })

df_validation = pd.DataFrame(csv_rows)
df_validation.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

print(f"Saved: {OUTPUT_CSV}")
print("\nRows:", len(df_validation))


Saved: validation_manual_review.csv

Rows: 192
